# Google Drive Connector for ADK + Gemini Enterprise (RAG-ready)
This notebook creates a **Google Drive connector** you can use in an ADK-based agent that will be hosted on **Vertex AI Agent Engine** and surfaced through **Gemini Enterprise**.

It includes:
- A Drive connector (`DriveConnector`) that can list/search files and extract text from common Drive types.
- Auth helpers for **local development** *and* **Gemini Enterprise token-injection**.
- Optional utilities to create a Google Doc from text and upload a PDF to Drive.
- A thin ADK tool wrapper layer (so the model can call the connector).

At the end, the notebook writes the connector modules into your repo under:
`src/research_agent/tools/`.


## 0) Prerequisites
Before running this notebook end-to-end:

### For local development testing
1. In Google Cloud Console, enable APIs:
   - **Google Drive API**
   - (Optional) **Google Docs API** (only needed if you will create Docs)
2. Create an OAuth Client ID (Desktop app is easiest for a notebook).
3. Download the OAuth client JSON (often named `client_secret_....json`).

### For Gemini Enterprise production
Gemini Enterprise will run the OAuth flow and then inject the resulting **access token** into the ADK session state.
To make that work, you’ll need:
1. An **Authorization resource** in Gemini Enterprise (Discovery Engine) for Google Drive scopes.
2. The `authorizationId` you used there.
3. Your Agent Engine deployment must set an env var: `GEMINI_ENTERPRISE_AUTH_ID=<authorizationId>`.

> Note: In Gemini Enterprise, the token is typically short-lived. Refresh is handled by the platform. If the token expires, the user may be prompted to re-consent.


## 1) Install dependencies
Run this once in your environment. (If you're inside a managed notebook environment, you may need a restart after installs.)


In [11]:
# If you're using a fresh environment, install required libs.
# You can comment out packages you already have.
!pip install "google-api-python-client>=2.0" "google-auth>=2.0" "google-auth-oauthlib>=1.0" "google-auth-httplib2>=0.1"                    "pypdf>=4.0" "reportlab>=4.0" "python-dotenv>=1.0"
%pip install -U google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client
%pip install -U pypdf
# Optional (only if you will wrap this as ADK tools in the same environment):
# !pip -q install "google-adk>=1.0.0"



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Users\smoreno\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached pypdf-6.7.5-py3-none-any.whl.metadata (7.1 kB)
Using cached pypdf-6.7.5-py3-none-any.whl (331 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Configure paths and scopes
Set the following variables:
- `PROJECT_ROOT`: repo root (where `src/` exists)
- `OAUTH_CLIENT_JSON`: path to your downloaded OAuth client JSON (local dev only)
- `TOKEN_CACHE`: where to cache OAuth tokens for local dev


In [1]:
from pathlib import Path

# Adjust if needed:
PROJECT_ROOT = Path(".").resolve()

# Local-dev OAuth client JSON (Desktop app recommended for notebooks).
# If you won't test locally, you can leave this unset.
OAUTH_CLIENT_JSON = PROJECT_ROOT / "client_secret.json"

# Local token cache (created after first auth):
TOKEN_CACHE = PROJECT_ROOT / ".cache" / "drive_token.json"
TOKEN_CACHE.parent.mkdir(parents=True, exist_ok=True)

DRIVE_SCOPES = [
    "https://www.googleapis.com/auth/drive.readonly",
    # Uncomment if you want to create / upload files in Drive:
     "https://www.googleapis.com/auth/drive.file",
    # Uncomment if using Docs API to write to documents:
     "https://www.googleapis.com/auth/documents",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OAUTH_CLIENT_JSON exists?:", OAUTH_CLIENT_JSON.exists())
print("TOKEN_CACHE:", TOKEN_CACHE)


PROJECT_ROOT: C:\Endava\EndevLocal\Research-Agent\notebooks
OAUTH_CLIENT_JSON exists?: False
TOKEN_CACHE: C:\Endava\EndevLocal\Research-Agent\notebooks\.cache\drive_token.json


## 3) Authentication helpers
This notebook supports two auth modes:

### A) Local development auth (Notebook)
Uses `google-auth-oauthlib` to open a browser consent flow and caches the token.

### B) Gemini Enterprise injected token (Production)
In production, Gemini Enterprise injects a user access token into the ADK `tool_context.state` under the key equal to your `authorizationId`. Your connector should first look for that token.

In this notebook we also provide a helper to create credentials from a raw access token to simulate that mode.


In [2]:
from __future__ import annotations
import json
from typing import Optional, Dict, Any

from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request

def creds_from_local_oauth(
    client_secrets_path,
    token_cache_path,
    scopes,
) -> Credentials:
    """Local-dev OAuth flow suitable for notebooks (Desktop app auth)."""
    from google_auth_oauthlib.flow import InstalledAppFlow

    creds: Optional[Credentials] = None
    if token_cache_path.exists():
        creds = Credentials.from_authorized_user_file(str(token_cache_path), scopes=scopes)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not client_secrets_path.exists():
                raise FileNotFoundError(
                    f"OAuth client secrets not found at {client_secrets_path}. "
                    "Download it from Google Cloud Console and set OAUTH_CLIENT_JSON."
                )
            flow = InstalledAppFlow.from_client_secrets_file(str(client_secrets_path), scopes=scopes)
            # This starts a local server; in some notebook environments you may need:
            # flow.run_console() instead of run_local_server().
            creds = flow.run_local_server(port=0)

        token_cache_path.parent.mkdir(parents=True, exist_ok=True)
        token_cache_path.write_text(creds.to_json(), encoding="utf-8")

    return creds

def creds_from_access_token(access_token: str, scopes) -> Credentials:
    """Create a Credentials object from an injected short-lived access token."""
    # Note: This credential won't be able to refresh unless a refresh_token is also provided.
    return Credentials(token=access_token, scopes=scopes)

# --- Example: get creds locally (uncomment to run) ---
# local_creds = creds_from_local_oauth(OAUTH_CLIENT_JSON, TOKEN_CACHE, DRIVE_SCOPES)
# print("Local creds acquired. Token expires?", local_creds.expired)


## 4) Implement the Drive connector
The connector below:
- Uses Drive API v3 for listing/searching and for downloading/exporting file content.
- Extracts text from:
  - Google Docs (`export text/plain`)
  - Google Sheets (`export text/csv`)
  - Google Slides (`export text/plain` when possible; otherwise exports PDF and extracts text)
  - PDFs (download + `pypdf` extraction)
  - Plain text files

It returns **structured objects** (metadata + text) that can be fed to a chunker/embedder for RAG.


In [3]:
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Optional, Dict, Any, Iterable
import io
import mimetypes

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload
from google.oauth2.credentials import Credentials

from pypdf import PdfReader

GOOGLE_DOC = "application/vnd.google-apps.document"
GOOGLE_SHEET = "application/vnd.google-apps.spreadsheet"
GOOGLE_SLIDE = "application/vnd.google-apps.presentation"
GOOGLE_FOLDER = "application/vnd.google-apps.folder"

@dataclass
class DriveFile:
    id: str
    name: str
    mimeType: str
    modifiedTime: Optional[str] = None
    webViewLink: Optional[str] = None

@dataclass
class DriveTextDocument(DriveFile):
    text: str = ""

class DriveConnector:
    def __init__(self, creds: Credentials):
        self.creds = creds
        self.drive = build("drive", "v3", credentials=creds, cache_discovery=False)

    def list_files(self, page_size: int = 10, include_folders: bool = False) -> List[DriveFile]:
        q = None if include_folders else f"mimeType != '{GOOGLE_FOLDER}'"
        resp = self.drive.files().list(
            q=q,
            pageSize=page_size,
            fields="files(id,name,mimeType,modifiedTime,webViewLink)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        return [DriveFile(**f) for f in resp.get("files", [])]

    def search_files(self, query: str, page_size: int = 10, include_folders: bool = False) -> List[DriveFile]:
        # Drive query language: https://developers.google.com/drive/api/guides/search-files
        q_parts = [query]
        if not include_folders:
            q_parts.append(f"mimeType != '{GOOGLE_FOLDER}'")
        q = " and ".join([p for p in q_parts if p])

        resp = self.drive.files().list(
            q=q,
            pageSize=page_size,
            fields="files(id,name,mimeType,modifiedTime,webViewLink)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        return [DriveFile(**f) for f in resp.get("files", [])]

    def _download_bytes(self, file_id: str) -> bytes:
        request = self.drive.files().get_media(fileId=file_id)
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
        return fh.getvalue()

    def _export_bytes(self, file_id: str, export_mime: str) -> bytes:
        # Export works only for Google Workspace docs (Docs/Sheets/Slides)
        data = self.drive.files().export(fileId=file_id, mimeType=export_mime).execute()
        return data if isinstance(data, (bytes, bytearray)) else bytes(str(data), "utf-8")

    def _extract_text_from_pdf_bytes(self, pdf_bytes: bytes) -> str:
        try:
            reader = PdfReader(io.BytesIO(pdf_bytes))
            parts = []
            for page in reader.pages:
                t = page.extract_text() or ""
                if t.strip():
                    parts.append(t)
            return "\n\n".join(parts).strip()
        except Exception as e:
            return f"[PDF_TEXT_EXTRACTION_FAILED: {type(e).__name__}: {e}]"

    def get_file_text(self, file_id: str) -> DriveTextDocument:
        meta = self.drive.files().get(
            fileId=file_id,
            fields="id,name,mimeType,modifiedTime,webViewLink",
            supportsAllDrives=True,
        ).execute()
        mime = meta.get("mimeType", "")
        name = meta.get("name", "")

        text = ""
        if mime == GOOGLE_DOC:
            text = self._export_bytes(file_id, "text/plain").decode("utf-8", errors="ignore")

        elif mime == GOOGLE_SHEET:
            text = self._export_bytes(file_id, "text/csv").decode("utf-8", errors="ignore")

        elif mime == GOOGLE_SLIDE:
            # Slides export: text/plain may not always be supported; PDF is a safe fallback.
            try:
                text = self._export_bytes(file_id, "text/plain").decode("utf-8", errors="ignore")
            except Exception:
                pdf_bytes = self._export_bytes(file_id, "application/pdf")
                text = self._extract_text_from_pdf_bytes(pdf_bytes)

        else:
            # Non-Google file: download raw
            raw = self._download_bytes(file_id)

            # Try PDF
            if mime == "application/pdf" or (name.lower().endswith(".pdf")):
                text = self._extract_text_from_pdf_bytes(raw)
            else:
                # Try plain decode (best-effort)
                text = raw.decode("utf-8", errors="ignore")

        return DriveTextDocument(
            id=meta["id"],
            name=meta.get("name", ""),
            mimeType=mime,
            modifiedTime=meta.get("modifiedTime"),
            webViewLink=meta.get("webViewLink"),
            text=text,
        )

    # --- Optional: write-back helpers ---

    def create_google_doc_from_text(self, title: str, content: str, folder_id: Optional[str] = None) -> DriveFile:
        # Create a blank Google Doc via Drive
        file_metadata = {"name": title, "mimeType": GOOGLE_DOC}
        if folder_id:
            file_metadata["parents"] = [folder_id]

        created = self.drive.files().create(
            body=file_metadata,
            fields="id,name,mimeType,modifiedTime,webViewLink",
            supportsAllDrives=True,
        ).execute()

        # Insert text via Docs API (requires documents scope and Docs API enabled)
        docs = build("docs", "v1", credentials=self.creds, cache_discovery=False)
        docs.documents().batchUpdate(
            documentId=created["id"],
            body={
                "requests": [
                    {"insertText": {"location": {"index": 1}, "text": content}}
                ]
            },
        ).execute()

        return DriveFile(**created)

    def upload_pdf_from_text(self, title: str, text: str, folder_id: Optional[str] = None) -> DriveFile:
        # Create a simple PDF from text using reportlab
        from reportlab.pdfgen import canvas
        from reportlab.lib.pagesizes import letter

        pdf_io = io.BytesIO()
        c = canvas.Canvas(pdf_io, pagesize=letter)
        width, height = letter

        y = height - 72
        for line in text.splitlines():
            c.drawString(72, y, line[:1200])
            y -= 14
            if y < 72:
                c.showPage()
                y = height - 72
        c.save()
        pdf_io.seek(0)

        file_metadata = {"name": f"{title}.pdf", "mimeType": "application/pdf"}
        if folder_id:
            file_metadata["parents"] = [folder_id]

        media = MediaIoBaseUpload(pdf_io, mimetype="application/pdf", resumable=False)

        created = self.drive.files().create(
            body=file_metadata,
            media_body=media,
            fields="id,name,mimeType,modifiedTime,webViewLink",
            supportsAllDrives=True,
        ).execute()

        return DriveFile(**created)


## 5) Quick local tests
Run these after you obtain local credentials. This validates the connector works *before* integrating into your agent.


In [4]:
#--- Acquire local creds (uncomment) ---
creds = creds_from_local_oauth(OAUTH_CLIENT_JSON, TOKEN_CACHE, DRIVE_SCOPES)
connector = DriveConnector(creds)

#--- List some files ---
files = connector.list_files(page_size=5)
for f in files:
    print(f)

#--- Search example (Drive query language) ---
results = connector.search_files("name contains 'report'", page_size=5)
for r in results:
    print(r)

#--- Extract text from first result ---
if results:
    doc = connector.get_file_text(results[0].id)
    print(doc.name, doc.mimeType)
    print(doc.text[:1000])


FileNotFoundError: OAuth client secrets not found at C:\Endava\EndevLocal\Research-Agent\notebooks\client_secret.json. Download it from Google Cloud Console and set OAUTH_CLIENT_JSON.

## 6) RAG-ready chunking (simple example)
A connector typically returns raw text; your RAG pipeline then chunks and embeds it.
Here is a simple chunker by character length with overlap.


In [ ]:
from typing import List, Dict

def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 150) -> List[str]:
    text = text or ""
    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")
    overlap = max(0, min(overlap, chunk_size - 1))

    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

# Example:
# chunks = chunk_text(doc.text)
# print(len(chunks), "chunks")


## 7) Wrap as ADK tools (for use in your agent)
In ADK, tools are usually `FunctionTool`s. In Gemini Enterprise production, the tool must pull the injected access token
from `tool_context.state[authorizationId]`.

This section provides a minimal tool wrapper pattern. You can paste the resulting modules into your repo.


In [ ]:
# NOTE: This section is a *template*. It requires the `google-adk` package.
# Uncomment the installs in section 1 if you want to run it here.

# from google.adk.tools import FunctionTool, ToolContext
from google.oauth2.credentials import Credentials

def get_creds_for_agent(tool_context, scopes, gemini_enterprise_auth_id: str | None):
    """Production-first credential getter.
    - If Gemini Enterprise injected a token, use it.
    - Else fall back to local token cache (dev).
    """
    if gemini_enterprise_auth_id:
        token = tool_context.state.get(gemini_enterprise_auth_id)
        if token:
            return Credentials(token=token, scopes=scopes)

    # DEV fallback:
    # return creds_from_local_oauth(OAUTH_CLIENT_JSON, TOKEN_CACHE, scopes)

    raise RuntimeError(
        "No credentials found. In production, set GEMINI_ENTERPRISE_AUTH_ID and ensure user OAuth is configured. "
        "In local dev, enable the DEV fallback."
    )

# Example tool function (ADK will inject tool_context):
def drive_search_files_tool(query: str, max_files: int, tool_context):
    import os
    auth_id = os.getenv("GEMINI_ENTERPRISE_AUTH_ID")
    creds = get_creds_for_agent(tool_context, DRIVE_SCOPES, auth_id)
    conn = DriveConnector(creds)
    files = conn.search_files(query=query, page_size=max_files)
    return [f.__dict__ for f in files]

# In your agent.py:
# drive_search_tool = FunctionTool(func=drive_search_files_tool)


## 8) Write the connector into your repo (creates modules under `src/research_agent/tools/`)
This cell writes two Python modules:
- `drive_connector.py` (connector implementation)
- `drive_auth_helpers.py` (credential helpers)

Adjust `PROJECT_ROOT` if your repo layout differs.


In [ ]:
import textwrap
from pathlib import Path

tools_dir = PROJECT_ROOT / "src" / "research_agent" / "tools"
tools_dir.mkdir(parents=True, exist_ok=True)

(tools_dir / "drive_auth_helpers.py").write_text(
    textwrap.dedent('''        """Authentication helpers for Google Drive.

    - Local dev: OAuth installed app flow (token cached on disk)
    - Production (Gemini Enterprise): access token injected into ADK session state

    Keep production tokens out of disk; rely on ToolContext state.
    """

    from __future__ import annotations
    from typing import Optional
    from pathlib import Path

    from google.oauth2.credentials import Credentials
    from google.auth.transport.requests import Request

    def creds_from_local_oauth(client_secrets_path: Path, token_cache_path: Path, scopes) -> Credentials:
        from google_auth_oauthlib.flow import InstalledAppFlow

        creds = None
        if token_cache_path.exists():
            creds = Credentials.from_authorized_user_file(str(token_cache_path), scopes=scopes)

        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                if not client_secrets_path.exists():
                    raise FileNotFoundError(
                        f"OAuth client secrets not found at {client_secrets_path}. "
                        "Download it from Google Cloud Console."
                    )
                flow = InstalledAppFlow.from_client_secrets_file(str(client_secrets_path), scopes=scopes)
                creds = flow.run_local_server(port=0)

            token_cache_path.parent.mkdir(parents=True, exist_ok=True)
            token_cache_path.write_text(creds.to_json(), encoding="utf-8")

        return creds

    def creds_from_access_token(access_token: str, scopes) -> Credentials:
        return Credentials(token=access_token, scopes=scopes)
    '''),
    encoding="utf-8"
)

(tools_dir / "drive_connector.py").write_text(
    textwrap.dedent('''        from __future__ import annotations
    from dataclasses import dataclass
    from typing import List, Optional
    import io

    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload
    from google.oauth2.credentials import Credentials

    from pypdf import PdfReader

    GOOGLE_DOC = "application/vnd.google-apps.document"
    GOOGLE_SHEET = "application/vnd.google-apps.spreadsheet"
    GOOGLE_SLIDE = "application/vnd.google-apps.presentation"
    GOOGLE_FOLDER = "application/vnd.google-apps.folder"

    @dataclass
    class DriveFile:
        id: str
        name: str
        mimeType: str
        modifiedTime: Optional[str] = None
        webViewLink: Optional[str] = None

    @dataclass
    class DriveTextDocument(DriveFile):
        text: str = ""

    class DriveConnector:
        def __init__(self, creds: Credentials):
            self.creds = creds
            self.drive = build("drive", "v3", credentials=creds, cache_discovery=False)

        def list_files(self, page_size: int = 10, include_folders: bool = False) -> List[DriveFile]:
            q = None if include_folders else f"mimeType != '{GOOGLE_FOLDER}'"
            resp = self.drive.files().list(
                q=q,
                pageSize=page_size,
                fields="files(id,name,mimeType,modifiedTime,webViewLink)",
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            ).execute()
            return [DriveFile(**f) for f in resp.get("files", [])]

        def search_files(self, query: str, page_size: int = 10, include_folders: bool = False) -> List[DriveFile]:
            q_parts = [query]
            if not include_folders:
                q_parts.append(f"mimeType != '{GOOGLE_FOLDER}'")
            q = " and ".join([p for p in q_parts if p])

            resp = self.drive.files().list(
                q=q,
                pageSize=page_size,
                fields="files(id,name,mimeType,modifiedTime,webViewLink)",
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            ).execute()
            return [DriveFile(**f) for f in resp.get("files", [])]

        def _download_bytes(self, file_id: str) -> bytes:
            request = self.drive.files().get_media(fileId=file_id)
            fh = io.BytesIO()
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
            return fh.getvalue()

        def _export_bytes(self, file_id: str, export_mime: str) -> bytes:
            data = self.drive.files().export(fileId=file_id, mimeType=export_mime).execute()
            return data if isinstance(data, (bytes, bytearray)) else bytes(str(data), "utf-8")

        def _extract_text_from_pdf_bytes(self, pdf_bytes: bytes) -> str:
            try:
                reader = PdfReader(io.BytesIO(pdf_bytes))
                parts = []
                for page in reader.pages:
                    t = page.extract_text() or ""
                    if t.strip():
                        parts.append(t)
                return "\n\n".join(parts).strip()
            except Exception as e:
                return f"[PDF_TEXT_EXTRACTION_FAILED: {type(e).__name__}: {e}]"

        def get_file_text(self, file_id: str) -> DriveTextDocument:
            meta = self.drive.files().get(
                fileId=file_id,
                fields="id,name,mimeType,modifiedTime,webViewLink",
                supportsAllDrives=True,
            ).execute()
            mime = meta.get("mimeType", "")
            name = meta.get("name", "")

            text = ""
            if mime == GOOGLE_DOC:
                text = self._export_bytes(file_id, "text/plain").decode("utf-8", errors="ignore")

            elif mime == GOOGLE_SHEET:
                text = self._export_bytes(file_id, "text/csv").decode("utf-8", errors="ignore")

            elif mime == GOOGLE_SLIDE:
                try:
                    text = self._export_bytes(file_id, "text/plain").decode("utf-8", errors="ignore")
                except Exception:
                    pdf_bytes = self._export_bytes(file_id, "application/pdf")
                    text = self._extract_text_from_pdf_bytes(pdf_bytes)

            else:
                raw = self._download_bytes(file_id)
                if mime == "application/pdf" or name.lower().endswith(".pdf"):
                    text = self._extract_text_from_pdf_bytes(raw)
                else:
                    text = raw.decode("utf-8", errors="ignore")

            return DriveTextDocument(
                id=meta["id"],
                name=meta.get("name", ""),
                mimeType=mime,
                modifiedTime=meta.get("modifiedTime"),
                webViewLink=meta.get("webViewLink"),
                text=text,
            )

        def create_google_doc_from_text(self, title: str, content: str, folder_id: Optional[str] = None) -> DriveFile:
            file_metadata = {"name": title, "mimeType": GOOGLE_DOC}
            if folder_id:
                file_metadata["parents"] = [folder_id]

            created = self.drive.files().create(
                body=file_metadata,
                fields="id,name,mimeType,modifiedTime,webViewLink",
                supportsAllDrives=True,
            ).execute()

            docs = build("docs", "v1", credentials=self.creds, cache_discovery=False)
            docs.documents().batchUpdate(
                documentId=created["id"],
                body={"requests": [{"insertText": {"location": {"index": 1}, "text": content}}]},
            ).execute()

            return DriveFile(**created)

        def upload_pdf_from_text(self, title: str, text: str, folder_id: Optional[str] = None) -> DriveFile:
            from reportlab.pdfgen import canvas
            from reportlab.lib.pagesizes import letter

            pdf_io = io.BytesIO()
            c = canvas.Canvas(pdf_io, pagesize=letter)
            width, height = letter

            y = height - 72
            for line in text.splitlines():
                c.drawString(72, y, line[:1200])
                y -= 14
                if y < 72:
                    c.showPage()
                    y = height - 72
            c.save()
            pdf_io.seek(0)

            file_metadata = {"name": f"{title}.pdf", "mimeType": "application/pdf"}
            if folder_id:
                file_metadata["parents"] = [folder_id]

            media = MediaIoBaseUpload(pdf_io, mimetype="application/pdf", resumable=False)

            created = self.drive.files().create(
                body=file_metadata,
                media_body=media,
                fields="id,name,mimeType,modifiedTime,webViewLink",
                supportsAllDrives=True,
            ).execute()

            return DriveFile(**created)
    '''),
    encoding="utf-8"
)

print("Wrote:", tools_dir / "drive_auth_helpers.py")
print("Wrote:", tools_dir / "drive_connector.py")


## 9) Next steps (integrate into your ADK agent)
1. Import `DriveConnector` and build ADK tool functions that:
   - read `GEMINI_ENTERPRISE_AUTH_ID`
   - pull token from `tool_context.state[AUTH_ID]`
   - instantiate `DriveConnector(creds)`
2. Add the `FunctionTool`s to your agent’s tool list.
3. Deploy to Agent Engine.
4. Register the agent in Gemini Enterprise and attach the Drive authorization.

If you already have a RAG pipeline, connect this connector’s output (`DriveTextDocument`) to your chunk/embed/index steps.
